In [1]:
import warnings
import numpy as np
import pandas as pd
import random
import pickle
import os
from datetime import datetime
from sklearn.feature_selection import RFE, RFECV
from sklearn.ensemble import RandomForestClassifier

import torch
from torch.utils.data import DataLoader, Subset
from collections import defaultdict

from data_import import DataPreprocessor, SingleDietImpactDataset, DataLoaderManager, MIDataset, WrapperDataset
from MetS_prediction_model import MultiDiseasePredictor_Base, MultiDiseasePredictor_MI, MultiDiseasePredictor_Regularized
from train_eval_function import train_model

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

Device: cuda
Device: cuda
=== 데이터 로드 및 전처리 ===
Device: cuda


# 데이터 로드

In [2]:
preprocessor = DataPreprocessor('../data/total_again.xlsx')
train_df, val_df, test_df, feature_cols = preprocessor.process_all()

train_dataset = SingleDietImpactDataset(train_df)
val_dataset = SingleDietImpactDataset(val_df)
test_dataset = SingleDietImpactDataset(test_df)

mets_cols = ['Increased waist circumference', 'Elevated blood pressure', 'Impaired fasting glucose', 
             'Elevated triglycerides', 'Decreased HDL-C']

print(f"원본 피처 수: diet({len(train_dataset.diet_cols)}), demo({len(train_dataset.demo_cols)}), \
      life({len(train_dataset.life_cols)}), bio({len(train_dataset.bio_cols)}), \
        delta({len(train_dataset.delta_cols)})")

Device: cuda
=== 데이터 로드 및 전처리 ===
원본 피처 수: diet(19), demo(4),       life(3), bio(11),         delta(22)


In [3]:
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
disease_cols = ['Decreased HDL-C_delta', 'Increased waist circumference_delta', 
                'Impaired fasting glucose_delta', 'Elevated blood pressure_delta', 
                'Elevated triglycerides_delta']

## MI 기반 피처 선택 결과 로드


In [ ]:
def load_mi_features(filepath='../result/feature_selection_experiment/mi_selected_features.pkl'):
    with open(filepath, 'rb') as f:
        mi_selected_features = pickle.load(f)
    print("=== MI 피처 선택 결과 로드 ===")
    for disease_name, features in mi_selected_features.items():
        print(f"{disease_name}: {len(features)}개 피처")
    return mi_selected_features

mi_selected_features = load_mi_features()

=== MI 피처 선택 결과 로드 ===
Decreased HDL-C: 23개 피처
Increased waist circumference: 38개 피처
Impaired fasting glucose: 25개 피처
Elevated blood pressure: 31개 피처
Elevated triglycerides: 30개 피처


## 래퍼 방법

In [5]:
def RFE_feature_selection(df, disease_cols, n_features):
    all_features = [col for col in df.columns if col not in disease_cols]
    results = {}
    
    for disease_col in disease_cols:
        disease_name = disease_col.replace('_delta', '')
        X = df[all_features].values
        y = df[disease_col].values
        
        rf = RandomForestClassifier(n_estimators=50, random_state=42)
        rfe = RFE(estimator=rf, n_features_to_select=n_features, step=1)
        rfe.fit_transform(X, y)
        
        selected_features = [all_features[i] for i in range(len(all_features)) if rfe.support_[i]]
        results[disease_name] = selected_features
        print(f"{disease_name}: {len(selected_features)}개 피처 선택")
    
    return results

RFE_selected_features = RFE_feature_selection(full_df, disease_cols, n_features=30)

Decreased HDL-C: 30개 피처 선택
Increased waist circumference: 30개 피처 선택
Impaired fasting glucose: 30개 피처 선택
Elevated blood pressure: 30개 피처 선택
Elevated triglycerides: 30개 피처 선택


In [6]:
def RFECV_feature_selection(df, disease_cols):
    all_features = [col for col in df.columns if col not in disease_cols]
    results = {}
    
    for disease_col in disease_cols:
        disease_name = disease_col.replace('_delta', '')
        X = df[all_features].values
        y = df[disease_col].values
        
        rf = RandomForestClassifier(n_estimators=50, random_state=42)
        rfecv = RFECV(estimator=rf)
        rfecv.fit_transform(X, y)
        
        selected_features = [all_features[i] for i in range(len(all_features)) if rfecv.support_[i]]
        results[disease_name] = selected_features
        print(f"{disease_name}: {len(selected_features)}개 피처 선택")
    
    return results

RFECV_selected_features = RFECV_feature_selection(full_df, disease_cols)

Decreased HDL-C: 18개 피처 선택
Increased waist circumference: 27개 피처 선택
Impaired fasting glucose: 24개 피처 선택
Elevated blood pressure: 27개 피처 선택
Elevated triglycerides: 50개 피처 선택


In [7]:
def split_dataset_by_disease_custom(dataset):
    disease_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        disease = dataset[idx]['disease_name']
        disease_to_indices[disease].append(idx)
    return {disease: Subset(dataset, indices) for disease, indices in disease_to_indices.items()}

# 모델 학습

In [8]:
results = {}

## 1. 기본 모델 (모든 피처)

In [9]:
print("\n=== 기본 모델 학습 ===")
train_loaders, val_loaders, test_loaders = DataLoaderManager.create_disease_loaders(
    train_df, val_df, test_df, batch_size=32
)

results['base'] = {}
for disease_name in mets_cols:
    print(f"\n{disease_name} - 기본 모델")
    model = MultiDiseasePredictor_Base(
        diet_dim=len(train_dataset.diet_cols),
        demo_dim=len(train_dataset.demo_cols),
        life_dim=len(train_dataset.life_cols),
        bio_dim=len(train_dataset.bio_cols),
        change_dim=len(train_dataset.delta_cols),
        disease_names=[disease_name]
    ).to(device)
    
    result = train_model(model, train_loaders[disease_name], val_loaders[disease_name], 
                        test_loaders[disease_name], disease_name, 'base')
    results['base'][disease_name] = result


=== 기본 모델 학습 ===

Increased waist circumference - 기본 모델
   Epoch 0: Train=1.3275, Val=1.1752
   Epoch 20: Train=0.4850, Val=0.4632
   Epoch 40: Train=0.4555, Val=0.4352
   Epoch 60: Train=0.4278, Val=0.4171
   Epoch 80: Train=0.4217, Val=0.4088

Elevated blood pressure - 기본 모델
   Epoch 0: Train=1.0386, Val=0.9717
   Epoch 20: Train=0.5678, Val=0.5232
   Epoch 40: Train=0.5299, Val=0.4858
   Epoch 60: Train=0.5145, Val=0.4608
   Epoch 80: Train=0.5068, Val=0.4576
   Early stopping at epoch 92

Impaired fasting glucose - 기본 모델
   Epoch 0: Train=1.3463, Val=1.1640
   Epoch 20: Train=0.7116, Val=0.6770
   Epoch 40: Train=0.6746, Val=0.6445
   Epoch 60: Train=0.6360, Val=0.6122
   Epoch 80: Train=0.6106, Val=0.5703

Elevated triglycerides - 기본 모델
   Epoch 0: Train=0.8817, Val=0.8175
   Epoch 20: Train=0.5528, Val=0.5343
   Epoch 40: Train=0.5258, Val=0.5018
   Epoch 60: Train=0.5085, Val=0.4826
   Epoch 80: Train=0.4993, Val=0.4818

Decreased HDL-C - 기본 모델
   Epoch 0: Train=0.9387, Val=0.9

## 2. MI 기반 모델


In [10]:
print("\n=== MI 기반 모델 학습 ===")
mi_train_dataset = MIDataset(train_df, mi_selected_features)
mi_val_dataset = MIDataset(val_df, mi_selected_features)
mi_test_dataset = MIDataset(test_df, mi_selected_features)

mi_train_disease_datasets = split_dataset_by_disease_custom(mi_train_dataset)
mi_val_disease_datasets = split_dataset_by_disease_custom(mi_val_dataset)
mi_test_disease_datasets = split_dataset_by_disease_custom(mi_test_dataset)

mi_train_loaders = {disease: DataLoader(ds, batch_size=32, shuffle=True) 
                    for disease, ds in mi_train_disease_datasets.items()}
mi_val_loaders = {disease: DataLoader(ds, batch_size=32, shuffle=False) 
                    for disease, ds in mi_val_disease_datasets.items()}
mi_test_loaders = {disease: DataLoader(ds, batch_size=32, shuffle=False) 
                    for disease, ds in mi_test_disease_datasets.items()}

results['mi'] = {}
for disease_name in mets_cols:
    print(f"\n{disease_name} - MI 모델")
    model = MultiDiseasePredictor_MI(
        selected_features_dict=mi_selected_features,
        disease_names=[disease_name]
    ).to(device)
    
    result = train_model(model, mi_train_loaders[disease_name], mi_val_loaders[disease_name], 
                        mi_test_loaders[disease_name], disease_name, 'mi')
    results['mi'][disease_name] = result


=== MI 기반 모델 학습 ===

Increased waist circumference - MI 모델
   Epoch 0: Train=0.9116, Val=0.8698
   Epoch 20: Train=0.4919, Val=0.4956
   Epoch 40: Train=0.4679, Val=0.4685
   Epoch 60: Train=0.4366, Val=0.4468
   Epoch 80: Train=0.4302, Val=0.4333

Elevated blood pressure - MI 모델
   Epoch 0: Train=1.1363, Val=1.0856
   Epoch 20: Train=0.5928, Val=0.5429
   Epoch 40: Train=0.5505, Val=0.5173
   Epoch 60: Train=0.5356, Val=0.4871
   Epoch 80: Train=0.5195, Val=0.4719
   Early stopping at epoch 95

Impaired fasting glucose - MI 모델
   Epoch 0: Train=1.0060, Val=0.9661
   Epoch 20: Train=0.7000, Val=0.6814
   Epoch 40: Train=0.6760, Val=0.6643
   Epoch 60: Train=0.6481, Val=0.6254
   Epoch 80: Train=0.6278, Val=0.5730

Elevated triglycerides - MI 모델
   Epoch 0: Train=1.1259, Val=1.0710
   Epoch 20: Train=0.5924, Val=0.5724
   Epoch 40: Train=0.5525, Val=0.5371
   Epoch 60: Train=0.5300, Val=0.5138
   Epoch 80: Train=0.5142, Val=0.4894

Decreased HDL-C - MI 모델
   Epoch 0: Train=1.0833, Val=

## 3. 래퍼 방법 모델

In [9]:
print("\\n=== RFE 방법 모델 학습 ===")
RFE_train_dataset = WrapperDataset(train_df, RFE_selected_features)
RFE_val_dataset = WrapperDataset(val_df, RFE_selected_features)
RFE_test_dataset = WrapperDataset(test_df, RFE_selected_features)

RFE_train_disease_datasets = split_dataset_by_disease_custom(RFE_train_dataset)
RFE_val_disease_datasets = split_dataset_by_disease_custom(RFE_val_dataset)
RFE_test_disease_datasets = split_dataset_by_disease_custom(RFE_test_dataset)

# 배치 크기를 16으로 줄여서 배치 불일치 문제 해결
RFE_train_loaders = {disease: DataLoader(ds, batch_size=16, shuffle=True, drop_last=True) 
                    for disease, ds in RFE_train_disease_datasets.items()}
RFE_val_loaders = {disease: DataLoader(ds, batch_size=16, shuffle=False, drop_last=False) 
                  for disease, ds in RFE_val_disease_datasets.items()}
RFE_test_loaders = {disease: DataLoader(ds, batch_size=16, shuffle=False, drop_last=False) 
                   for disease, ds in RFE_test_disease_datasets.items()}

results['RFE'] = {}
for disease_name in mets_cols:
    print(f"\\n{disease_name} - RFE 모델")
    model = MultiDiseasePredictor_MI(
        selected_features_dict=RFE_selected_features,
        disease_names=[disease_name]
    ).to(device)
    
    result = train_model(model, RFE_train_loaders[disease_name], RFE_val_loaders[disease_name], 
                        RFE_test_loaders[disease_name], disease_name, 'RFE')
    results['RFE'][disease_name] = result

\n=== RFE 방법 모델 학습 ===
\nIncreased waist circumference - RFE 모델
   Epoch 0: Train=1.1588, Val=0.9966
   Epoch 20: Train=0.4944, Val=0.4980
   Epoch 40: Train=0.4668, Val=0.4772
   Epoch 60: Train=0.4490, Val=0.4526
   Epoch 80: Train=0.4370, Val=0.4417
\nElevated blood pressure - RFE 모델
   Epoch 0: Train=0.9603, Val=0.8791
   Epoch 20: Train=0.5696, Val=0.5442
   Epoch 40: Train=0.5400, Val=0.5004
   Epoch 60: Train=0.5235, Val=0.4861
   Epoch 80: Train=0.5193, Val=0.4827
   Early stopping at epoch 84
\nImpaired fasting glucose - RFE 모델
   Epoch 0: Train=0.8822, Val=0.8213
   Epoch 20: Train=0.7025, Val=0.6693
   Epoch 40: Train=0.6801, Val=0.6482
   Epoch 60: Train=0.6488, Val=0.6182
   Epoch 80: Train=0.6302, Val=0.5742
\nElevated triglycerides - RFE 모델
   Epoch 0: Train=1.1183, Val=1.0573
   Epoch 20: Train=0.5748, Val=0.5898
   Epoch 40: Train=0.5465, Val=0.5588
   Epoch 60: Train=0.5302, Val=0.5391
   Epoch 80: Train=0.5301, Val=0.5280
\nDecreased HDL-C - RFE 모델
   Epoch 0: Train=

In [10]:
print("\n=== RFECV 방법 모델 학습 ===")
RFECV_train_dataset = WrapperDataset(train_df, RFECV_selected_features)
RFECV_val_dataset = WrapperDataset(val_df, RFECV_selected_features)
RFECV_test_dataset = WrapperDataset(test_df, RFECV_selected_features)

RFECV_train_disease_datasets = split_dataset_by_disease_custom(RFECV_train_dataset)
RFECV_val_disease_datasets = split_dataset_by_disease_custom(RFECV_val_dataset)
RFECV_test_disease_datasets = split_dataset_by_disease_custom(RFECV_test_dataset)

RFECV_train_loaders = {disease: DataLoader(ds, batch_size=32, shuffle=True) 
                        for disease, ds in RFECV_train_disease_datasets.items()}
RFECV_val_loaders = {disease: DataLoader(ds, batch_size=32, shuffle=False) 
                      for disease, ds in RFECV_val_disease_datasets.items()}
RFECV_test_loaders = {disease: DataLoader(ds, batch_size=32, shuffle=False) 
                       for disease, ds in RFECV_test_disease_datasets.items()}

results['RFECV'] = {}
for disease_name in mets_cols:
    print(f"\n{disease_name} - 래퍼 모델")
    model = MultiDiseasePredictor_MI(
        selected_features_dict=RFECV_selected_features,
        disease_names=[disease_name]
    ).to(device)
    
    result = train_model(model, RFECV_train_loaders[disease_name], RFECV_val_loaders[disease_name], 
                        RFECV_test_loaders[disease_name], disease_name, 'RFECV')
    results['RFECV'][disease_name] = result


=== RFECV 방법 모델 학습 ===

Increased waist circumference - 래퍼 모델
   Epoch 0: Train=1.1467, Val=1.0271
   Epoch 20: Train=0.5128, Val=0.4946
   Epoch 40: Train=0.4754, Val=0.4677
   Epoch 60: Train=0.4531, Val=0.4470
   Epoch 80: Train=0.4356, Val=0.4311

Elevated blood pressure - 래퍼 모델
   Epoch 0: Train=1.1203, Val=0.9598
   Epoch 20: Train=0.6010, Val=0.5504
   Epoch 40: Train=0.5602, Val=0.5262
   Epoch 60: Train=0.5516, Val=0.5148
   Early stopping at epoch 74

Impaired fasting glucose - 래퍼 모델
   Epoch 0: Train=0.9812, Val=0.9589
   Epoch 20: Train=0.7129, Val=0.6911
   Epoch 40: Train=0.6928, Val=0.6737
   Epoch 60: Train=0.6735, Val=0.6534
   Epoch 80: Train=0.6546, Val=0.6336

Elevated triglycerides - 래퍼 모델
   Epoch 0: Train=0.8582, Val=0.8898
   Epoch 20: Train=0.5854, Val=0.5985
   Epoch 40: Train=0.5612, Val=0.5675
   Epoch 60: Train=0.5438, Val=0.5462
   Epoch 80: Train=0.5257, Val=0.5225

Decreased HDL-C - 래퍼 모델
   Epoch 0: Train=0.9163, Val=0.8426
   Epoch 20: Train=0.5078, V

## 4. 정규화 모델

In [12]:
print("\n=== 정규화 모델 학습 ===")
results['regularized'] = {}
for disease_name in mets_cols:
    print(f"\n{disease_name} - 정규화 모델")
    model = MultiDiseasePredictor_Regularized(
        diet_dim=len(train_dataset.diet_cols),
        demo_dim=len(train_dataset.demo_cols),
        life_dim=len(train_dataset.life_cols),
        bio_dim=len(train_dataset.bio_cols),
        change_dim=len(train_dataset.delta_cols),
        disease_names=[disease_name],
        l1_lambda=0.01,
        l2_lambda=0.01
    ).to(device)
    
    result = train_model(model, train_loaders[disease_name], val_loaders[disease_name], 
                        test_loaders[disease_name], disease_name, 'regularized')
    results['regularized'][disease_name] = result


=== 정규화 모델 학습 ===

Increased waist circumference - 정규화 모델
   Epoch 0: Train=3.4658, Val=0.8880
   Epoch 20: Train=1.0831, Val=0.4972
   Epoch 40: Train=0.6614, Val=0.4780
   Early stopping at epoch 45

Elevated blood pressure - 정규화 모델
   Epoch 0: Train=3.7539, Val=1.0855
   Epoch 20: Train=1.1470, Val=0.5323
   Epoch 40: Train=0.7479, Val=0.5280
   Early stopping at epoch 43

Impaired fasting glucose - 정규화 모델
   Epoch 0: Train=3.5135, Val=0.8907
   Epoch 20: Train=1.2488, Val=0.6794
   Epoch 40: Train=0.8556, Val=0.6577
   Early stopping at epoch 41

Elevated triglycerides - 정규화 모델
   Epoch 0: Train=3.6518, Val=0.9828
   Epoch 20: Train=1.1681, Val=0.5599
   Epoch 40: Train=0.7486, Val=0.5584
   Early stopping at epoch 43

Decreased HDL-C - 정규화 모델
   Epoch 0: Train=3.7264, Val=1.0469
   Epoch 20: Train=1.1389, Val=0.4229
   Epoch 40: Train=0.6682, Val=0.4052
   Early stopping at epoch 46


# 결과 분석 및 시각화

In [11]:
performance_data = []

for method_name, method_results in results.items():
    for disease, disease_results in method_results.items():
        eval_results = disease_results['evaluation']
        performance_data.append({
            'Method': method_name,
            'Disease': disease,
            'Accuracy': eval_results['accuracy'],
            'F1': eval_results['f1_score'],
            'ROC_AUC': eval_results['roc_aucs']['Macro'],
            'PR_AUC': eval_results['pr_aucs']['Macro']
        })

performance_df = pd.DataFrame(performance_data)

metrics = ['Accuracy', 'F1', 'ROC_AUC', 'PR_AUC']
for metric in metrics:
    pivot = performance_df.pivot(index='Disease', columns='Method', values=metric)
    print(f"\n=== {metric} ===")
    print(pivot.round(3))

print("\n=== 질병별 최고 성능 방법 (PR_AUC 기준) ===")
best_methods = {}
for disease in mets_cols:
    disease_data = performance_df[performance_df['Disease'] == disease]
    best_idx = disease_data['PR_AUC'].idxmax()
    best_method = disease_data.loc[best_idx, 'Method']
    best_prauc = disease_data.loc[best_idx, 'PR_AUC']
    best_methods[disease] = {'method': best_method, 'PR_AUC': best_prauc}
    print(f"{disease}: {best_method} (PR_AUC: {best_prauc:.3f})")


=== Accuracy ===
Method                           RFE  RFECV
Disease                                    
Decreased HDL-C                0.848  0.848
Elevated blood pressure        0.831  0.831
Elevated triglycerides         0.819  0.819
Impaired fasting glucose       0.787  0.787
Increased waist circumference  0.860  0.860

=== F1 ===
Method                           RFE  RFECV
Disease                                    
Decreased HDL-C                0.306  0.306
Elevated blood pressure        0.303  0.303
Elevated triglycerides         0.300  0.300
Impaired fasting glucose       0.294  0.294
Increased waist circumference  0.308  0.308

=== ROC_AUC ===
Method                           RFE  RFECV
Disease                                    
Decreased HDL-C                0.839  0.842
Elevated blood pressure        0.774  0.689
Elevated triglycerides         0.796  0.785
Impaired fasting glucose       0.790  0.769
Increased waist circumference  0.793  0.794

=== PR_AUC ===
Method       

# 결과 저장

In [12]:
results_dir = '../result/feature_selection_experiment'
timestamp = datetime.now().strftime('%Y%m%d')
save_dir = os.path.join(results_dir, f'exp_{timestamp}')
os.makedirs(save_dir, exist_ok=True)

# 1. 성능 데이터프레임 저장
performance_df.to_csv(os.path.join(save_dir, 'performance_comparison.csv'), index=False)

# 2. 최고 성능 방법들 저장
with open(os.path.join(save_dir, 'best_methods.pkl'), 'wb') as f:
    pickle.dump(best_methods, f)

# 3. 선택된 피처들 저장
if mi_selected_features is not None:
    feature_selections = {
        'mi_selected': mi_selected_features,
        'RFE_selected': RFE_selected_features,
        'RFECV_selected':RFECV_selected_features
    }
else:
    feature_selections = {
        'wrapper_selected': RFE_selected_features,
        'RFECV_selected': RFECV_selected_features
        
    }
with open(os.path.join(save_dir, 'selected_features.pkl'), 'wb') as f:
    pickle.dump(feature_selections, f)

# 4. 최고 성능 모델들만 저장
best_models = {}
for disease in mets_cols:
    best_method = best_methods[disease]['method']
    best_models[disease] = results[best_method][disease]['model']
torch.save(best_models, os.path.join(save_dir, 'best_models.pth'))

# 5. 실험 설정 저장
experiment_config = {
    'methods': ['base', 'mi', 'RFE', 'RFECV', 'regularized'],
    'diseases': mets_cols,
    'mi_alpha': 0.05,
    'mi_top_k': 20,
    'wrapper_n_features': 30,
    'regularization': {'l1_lambda': 0.01, 'l2_lambda': 0.01},
    'timestamp': timestamp,
    'device': str(device)
}
with open(os.path.join(save_dir, 'experiment_config.pkl'), 'wb') as f:
    pickle.dump(experiment_config, f)

print(f"\n결과가 {save_dir}에 저장되었습니다.")


결과가 ../result/feature_selection_experiment\exp_20250804에 저장되었습니다.
